In [1]:
import torch
from transformers import AutoModel
import torch.nn as nn
import os

In [3]:
llm = AutoModel.from_pretrained(
    "./omnivoice_qwen3",
    dtype=torch.float32,   
)
llm.eval()
llm.to("cuda")
pass

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [4]:
embedding = llm.get_input_embeddings()   

dummy_input = torch.randint(
    0, 
    llm.config.vocab_size, 
    (2, 220), 
    dtype=torch.long
).to("cuda")

In [5]:
output_dir = "onnx_qwen3_input_embeddings"

In [6]:
os.makedirs(output_dir, exist_ok=True)

In [7]:
torch.onnx.export(
    embedding,
    dummy_input,
    os.path.join(output_dir, "model.onnx"),
    input_names=["input_ids"],
    output_names=["embeddings"],
    dynamic_axes={
        "input_ids": {0: "batch_size", 1: "seq_length"},
        "embeddings": {0: "batch_size", 1: "seq_length"},
    },
    opset_version=17,
    do_constant_folding=True,
    external_data=False,
)

/tmp/ipykernel_61763/3630046234.py:1: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1101 21:27:14.571000 61763 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W1101 21:27:16.880000 61763 site-packages/torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1101 21:27:16.881000 61763 site-packages/torch/onnx/_inter

[torch.onnx] Obtain model graph for `Embedding(151676, 1024)` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Embedding(151676, 1024)` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 17},
            producer_name='pytorch',
            producer_version='2.14.1+cu130',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input_ids"<INT64,[batch_size,seq_length]>
            ),
            outputs=(
                %"embeddings"<FLOAT,[batch_size,seq_length,1024]>
            ),
            initializers=(
                %"weight"<FLOAT,[151676,1024]>{TorchTensor(...)}
            ),
        ) {
            0 |  # node_embedding
                 %"embeddings"<FLOAT,[batch_size,seq_length,1024]> ⬅️ ::Gather(%"weight"{...}, %"input_ids") {axis=0}
            return %"embeddings"<FLOAT,[batch_size,seq_length,1024]>
        }


    ,
    exported_program=
        ExportedProgram:
            class GraphModule(torch.nn.Module):
                def forward(self, p_weight: "f32[151676, 1